In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
from pathlib import Path

csv_path = Path("Mutual_Fund_Nav.csv")
out_path = Path("Mutual_Fund_Nav.feather")

drop_cols = [
    "OPDATE",
    "OPMODE",
    "PRICE_DATE_VIRTUAL",
    "CRNCY_CODE",
    "OBJECT_ID",
    "F_NAV_ACCUMULATED",
]

dtype_map = {
    "F_INFO_WINDCODE": "string",
    "ANN_DATE": "int64",      # 关键：因为有 19981130.0 这种
    "PRICE_DATE": "int64",
    "F_NAV_UNIT": "float64",
    "F_NAV_DIVACCUMULATED": "float64",
    "F_NAV_ADJFACTOR": "float64",
    "F_PRT_NETASSET": "float64",
    "F_ASSET_MERGEDSHARESORNOT": "int64",
    "NETASSET_TOTAL": "float64",
    "F_NAV_ADJUSTED": "float64",
    "IS_EXDIVIDENDDATE": "float64",
    "F_NAV_DISTRIBUTION": "float64",
    "S_INFO_ASHARECODE": "string",
    "CUM_NET_ASSET_VALUE": "float64",
}

chunksize = 500_000
tables = []

for chunk in pd.read_csv(
    csv_path,
    chunksize=chunksize,
    low_memory=False,
    dtype=dtype_map
):
    chunk = chunk.drop(columns=[c for c in drop_cols if c in chunk.columns])
    tables.append(pa.Table.from_pandas(chunk, preserve_index=False))

table = pa.concat_tables(tables)
feather.write_feather(table, out_path)

print("转换完成")
print("行数：", table.num_rows)
print("列数：", table.num_columns)

转换完成
行数： 35376886
列数： 14


In [4]:
import pandas as pd

df = pd.read_feather("Mutual_Fund_Nav.feather")

df = df.dropna(subset=["ANN_DATE"])

df["ANN_DATE"] = df["ANN_DATE"].astype("int64")
print(df.dtypes)
df.reset_index(drop=True).to_feather("Mutual_Fund_Nav.feather")

F_INFO_WINDCODE              string[python]
ANN_DATE                              int64
PRICE_DATE                            int64
F_NAV_UNIT                          float64
F_NAV_DIVACCUMULATED                float64
F_NAV_ADJFACTOR                     float64
F_PRT_NETASSET                      float64
F_ASSET_MERGEDSHARESORNOT             int64
NETASSET_TOTAL                      float64
F_NAV_ADJUSTED                      float64
IS_EXDIVIDENDDATE                   float64
F_NAV_DISTRIBUTION                  float64
S_INFO_ASHARECODE            string[python]
CUM_NET_ASSET_VALUE                 float64
dtype: object


In [14]:
import pandas as pd
df1 = pd.read_csv('ASHAREINDUSTRIESCODE_202605221326.csv')
df2 = pd.read_csv('CHINAMUTUALFUNDSECTOR_202605221321.csv')
print(df1.columns)
print(df2.columns)
print(df1.head(1))
print(df2.head(1))

Index(['OBJECT_ID', 'INDUSTRIESCODE', 'INDUSTRIESNAME', 'LEVELNUM', 'USED',
       'INDUSTRIESALIAS', 'SEQUENCE', 'MEMO', 'CHINESEDEFINITION',
       'WIND_NAME_ENG', 'INDUSTRIESCODE_OLD', 'REGION_CODE', 'OPDATE',
       'OPMODE'],
      dtype='object')
Index(['OBJECT_ID', 'F_INFO_WINDCODE', 'S_INFO_SECTOR', 'S_INFO_SECTORENTRYDT',
       'S_INFO_SECTOREXITDT', 'CUR_SIGN', 'OPDATE', 'OPMODE'],
      dtype='object')
                                OBJECT_ID    INDUSTRIESCODE  \
0  {FDB644A6-C8B2-4B9B-B14B-64E5625570DD}  xx55100000000000   

         INDUSTRIESNAME  LEVELNUM  USED INDUSTRIESALIAS  SEQUENCE MEMO  \
0  MSCICHINAA/UTILITIES         3     1          133613       NaN  NaN   

  CHINESEDEFINITION         WIND_NAME_ENG INDUSTRIESCODE_OLD  REGION_CODE  \
0               NaN  MSCICHINAA/UTILITIES         xx55100000          NaN   

                    OPDATE  OPMODE  
0  2008-11-10 13:50:54.000       0  
                                OBJECT_ID F_INFO_WINDCODE S_INFO_SECTOR  \
0

In [ ]:
import pandas as pd
import numpy as np
import re


# 优先级（越靠前优先级越高）
FUND_TYPE_PRIORITY = [
    "FOF",
    "ETF",
    "LOF",
    "QDII",
    "Money Market Fund",
    "Bond Fund",
    "Equity Fund",      # 包含股票型 + 偏股混合 + 选股基金
    "Hybrid Fund",
    "Other/Unknown"
]


def classify_one_type(text: str,fund_code: str = "") -> str:

    if pd.isna(text):
        return "Other/Unknown"

    x = str(text).upper()
    fund_code = str(fund_code)

     
    # 1. FOF
    # ETF-FOF / Pension FOF / MOM 全部归 FOF
     
    if re.search(
        r"FOF|FUND OF FUNDS|PENSION|TARGET DATE|TARGET RISK|MOM|MANAGER OF MANAGERS",
        x
    ):
        return "FOF"

     
    # 2. ETF
     
    if ( 
        fund_code.startswith(tuple(["510","511","512","513","515","516","517","518","520","530","551","560","561","562","563","588","589","159"]))
        or
        re.search(
            r"\bETF\b|EXCHANGE TRADED FUND",
            x
        )
    ):
        return "ETF"

     
    # 3. LOF
     
    if re.search(
        r"\bLOF\b|LISTED OPEN[- ]?ENDED",
        x
    ):
        return "LOF"

     
    # 4. QDII
     
    if re.search(
        r"\bQDII\b|QUALIFIED DOMESTIC INSTITUTIONAL INVESTOR",
        x
    ):
        return "QDII"

     
    # 5. Money Market Fund
     
    if re.search(
        r"MONEY MARKET|MONEY FUND|CASH FUND",
        x
    ):
        return "Money Market Fund"

     
    # 6. Bond Fund
     
    if re.search(
        r"BOND FUND|BOND|FIXED INCOME|CONVERTIBLE BOND|SHORT[- ]?TERM BOND",
        x
    ):
        return "Bond Fund"

     
    # 7. Equity Fund
    #
    # 把：
    # - 股票型
    # - 偏股混合
    # - 选股基金
    # 全部合并
     
    if re.search(
        r"EQUITY|STOCK FUND|STOCK|EQUITY FUND|ACTIVE EQUITY|"
        r"AGGRESSIVE ALLOCATION|PARTIAL EQUITY|"
        r"FLEXIBLE ALLOCATION|SELECTION FUND|"
        r"SHARE FUND|GROWTH FUND",
        x
    ):
        return "Equity Fund"

     
    # 8. Hybrid Fund
    # 剩余混合型
     
    if re.search(
        r"HYBRID|BALANCED|MIXED|ALLOCATION FUND",
        x
    ):
        return "Hybrid Fund"

    return "Other/Unknown"


def classify_fund_main_type(industry_df,
                             fund_sector_df,
                             current_only=True):

    industry = industry_df.copy()
    fund_sector = fund_sector_df.copy()

    # 转字符串
    industry["INDUSTRIESCODE"] = industry["INDUSTRIESCODE"].astype(str)
    fund_sector["S_INFO_SECTOR"] = fund_sector["S_INFO_SECTOR"].astype(str)

    # 只保留有效分类
    if "USED" in industry.columns:
        industry = industry[industry["USED"] == 1]

    # 只保留当前分类
    if current_only and "CUR_SIGN" in fund_sector.columns:
        fund_sector = fund_sector[fund_sector["CUR_SIGN"] == 1]

    # 合并
    df = fund_sector.merge(
        industry,
        left_on="S_INFO_SECTOR",
        right_on="INDUSTRIESCODE",
        how="left"
    )

    # 可用于分类的文本字段
    text_cols = [
        "INDUSTRIESNAME",
        "INDUSTRIESALIAS",
        "MEMO",
        "CHINESEDEFINITION",
        "WIND_NAME_ENG"
    ]

    # 防止缺失字段
    for col in text_cols:
        if col not in df.columns:
            df[col] = ""

    # 拼接文本
    df["type_text"] = (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

    # 分类
    df["fund_type"] = df.apply(
    lambda row: classify_one_type(
        row["type_text"],
        row["F_INFO_WINDCODE"]
    ),
    axis=1
)

    # 优先级映射
    priority_map = {
        v: i for i, v in enumerate(FUND_TYPE_PRIORITY)
    }

    # 一个基金可能对应多个分类
    # 取优先级最高的那个
    def choose_main_type(types):

        types = list(set(types))

        if len(types) == 0:
            return "Other/Unknown"

        return sorted(
            types,
            key=lambda x: priority_map.get(x, 999)
        )[0]

    result = (
        df.groupby("F_INFO_WINDCODE")
        .agg(
            main_type=("fund_type", choose_main_type),

            all_types=(
                "fund_type",
                lambda x: sorted(set(x))
            ),

            matched_sector_names=(
                "INDUSTRIESNAME",
                lambda x: sorted(set(
                    x.dropna().astype(str)
                ))
            ),

            sector_codes=(
                "S_INFO_SECTOR",
                lambda x: sorted(set(
                    x.dropna().astype(str)
                ))
            )
        )
        .reset_index()
    )

    return result, df

In [96]:
fund_type_result, detail = classify_fund_main_type(
    industry_df=df1,
    fund_sector_df=df2,
    current_only=False
)

In [97]:
fund_type_result['main_type'].value_counts()

main_type
Equity Fund          15055
Bond Fund             9307
Hybrid Fund           2046
ETF                   1735
FOF                   1366
Money Market Fund     1172
QDII                   718
Other/Unknown          134
Name: count, dtype: int64

In [98]:
Stock_funds = fund_type_result[
    fund_type_result["main_type"] == "Equity Fund"
]
print(Stock_funds)
print(sum(Stock_funds["F_INFO_WINDCODE"].str.startswith(tuple(["510","511","512","513","515","516","517","518","520","530","551","560","561","562","563","588","589","159"]))))

      F_INFO_WINDCODE    main_type                     all_types  \
0           000001.OF  Equity Fund  [Equity Fund, Other/Unknown]   
4           000006.OF  Equity Fund  [Equity Fund, Other/Unknown]   
6           000008.OF  Equity Fund  [Equity Fund, Other/Unknown]   
9           000011.OF  Equity Fund  [Equity Fund, Other/Unknown]   
14          000017.OF  Equity Fund  [Equity Fund, Other/Unknown]   
...               ...          ...                           ...   
31526      F180003.OF  Equity Fund  [Equity Fund, Other/Unknown]   
31527      F202003.OF  Equity Fund  [Equity Fund, Other/Unknown]   
31528      F202017.OF  Equity Fund  [Equity Fund, Other/Unknown]   
31531      F450004.OF  Equity Fund  [Equity Fund, Other/Unknown]   
31532      F450005.OF  Equity Fund  [Equity Fund, Other/Unknown]   

      matched_sector_names                                       sector_codes  
0       [偏股混合型基金, 灵活配置型基金]  [2001010201000000, 2001010204000000, 200103010...  
4                [偏股混合型

In [1]:
import pandas as pd 
df = pd.read_feather('Mutual_Fund_Nav.feather')
df['ANN_DATE']

0           19981130
1           19981207
2           19981214
3           19981221
4           19981228
              ...   
35375157    20250621
35375158    20250422
35375159    20240111
35375160    20260303
35375161    20240518
Name: ANN_DATE, Length: 35375162, dtype: int64

In [14]:
import pandas as pd
df = pd.read_feather('D:\BaiduNetdiskDownload\正则化基金数据\正则化基金数据\基金数据\Mutual_Fund_Nav.feather')
df[(df['F_INFO_WINDCODE']=='002834.OF') & (df['PRICE_DATE']>20180220)]

,F_INFO_WINDCODE,ANN_DATE,PRICE_DATE,F_NAV_UNIT,F_NAV_DIVACCUMULATED,F_NAV_ADJFACTOR,F_PRT_NETASSET,F_ASSET_MERGEDSHARESORNOT,NETASSET_TOTAL,F_NAV_ADJUSTED,IS_EXDIVIDENDDATE,F_NAV_DISTRIBUTION,S_INFO_ASHARECODE,CUM_NET_ASSET_VALUE
5663867,002834.OF,20180421,20180331,1.1179,NaN,1.000000,NaN,0,NaN,1.117900,0.0,0.0,S6587098,1.1179
17445594,002834.OF,20230302,20230301,1.6327,NaN,0.700395,NaN,0,NaN,1.143534,0.0,0.0,S6587098,1.6327
17575921,002834.OF,20230218,20230217,1.5961,NaN,0.700395,NaN,0,NaN,1.117900,1.0,0.0,S6587098,1.5961
17585289,002834.OF,20230315,20230314,1.5811,NaN,0.700395,NaN,0,NaN,1.107394,0.0,0.0,S6587098,1.5811
17596898,002834.OF,20230317,20230316,1.5781,NaN,0.700395,NaN,0,NaN,1.105293,0.0,0.0,S6587098,1.5781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35160175,002834.OF,20241217,20241216,2.1825,NaN,0.700395,NaN,0,NaN,1.528611,0.0,0.0,S6587098,2.1825
35184247,002834.OF,20260331,20260330,3.0173,NaN,0.700395,NaN,0,NaN,2.113301,0.0,0.0,S6587098,3.0173
35201399,002834.OF,20250215,20250214,2.0617,NaN,0.700395,NaN,0,NaN,1.444004,0.0,0.0,S6587098,2.0617
35239640,002834.OF,20260514,20260513,3.2966,NaN,0.700395,NaN,0,NaN,2.308921,0.0,0.0,S6587098,3.2966


In [ ]:
# 前置诊断：检查原始 nav 的 PRICE_DATE 是否连续。
# 同一基金相邻 PRICE_DATE 间隔超过 max_shift_days，认为这里出现了一次不连续。
# 这里只统计，不剔除基金：
# - 如果只有少数日期不连续，保留基金，后续由 max_shift_days 对齐规则处理。
# - 如果一只基金形成两段或多段连续日期，也只统计出来，不在这里剔除。

nav_unique = nav_unique.sort_values([fund_col, price_date_col]).copy()
nav_unique['PRICE_DATE_GAP_DAYS'] = nav_unique.groupby(fund_col)[price_date_col].diff().dt.days

discontinuous_mask = nav_unique['PRICE_DATE_GAP_DAYS'] > max_shift_days
nav_unique['CONTINUOUS_SEGMENT_ID'] = discontinuous_mask.groupby(nav_unique[fund_col]).cumsum().astype(int)
discontinuous_funds = pd.Index(nav_unique.loc[discontinuous_mask, fund_col].dropna().unique())

continuity_summary = (
    nav_unique
    .groupby(fund_col)
    .agg(
        nav_rows=(price_date_col, 'size'),
        first_price_date=(price_date_col, 'min'),
        last_price_date=(price_date_col, 'max'),
        max_gap_days=('PRICE_DATE_GAP_DAYS', 'max'),
        discontinuous_gap_count=('PRICE_DATE_GAP_DAYS', lambda s: int((s > max_shift_days).sum())),
        continuous_segment_count=('CONTINUOUS_SEGMENT_ID', lambda s: int(s.max() + 1)),
    )
    .reset_index()
)

discontinuous_fund_summary = (
    continuity_summary
    .loc[continuity_summary['discontinuous_gap_count'] > 0]
    .sort_values(['discontinuous_gap_count', 'max_gap_days'], ascending=False)
)
multi_segment_fund_summary = (
    continuity_summary
    .loc[continuity_summary['continuous_segment_count'] >= 2]
    .sort_values(['continuous_segment_count', 'max_gap_days'], ascending=False)
)

print('原始 nav PRICE_DATE 出现不连续 gap 的基金数量:', len(discontinuous_funds))
print('原始 nav PRICE_DATE 不连续 gap 总次数:', int(discontinuous_mask.sum()))
print('存在两段或多段连续日期的基金数量:', len(multi_segment_fund_summary))

if len(discontinuous_funds) > 0:
    display(
        nav_unique.loc[discontinuous_mask, [fund_col, price_date_col, 'PRICE_DATE_GAP_DAYS', 'CONTINUOUS_SEGMENT_ID']]
        .sort_values('PRICE_DATE_GAP_DAYS', ascending=False)
        .head(30)
    )

print('\n不连续基金汇总 Top 30:')
display(discontinuous_fund_summary.head(30))

print('\n两段或多段连续日期基金汇总 Top 30:')
display(multi_segment_fund_summary.head(30))

# 后续对齐不需要这两个诊断辅助列，先从工作表中移除；summary 变量会保留。
nav_unique = nav_unique.drop(columns=['PRICE_DATE_GAP_DAYS', 'CONTINUOUS_SEGMENT_ID'])

In [1]:
import pandas as pd
nav = pd.read_feather('股票型_偏股混合型_nav.feather')
nav[(nav['F_INFO_WINDCODE']=='920003.OF') & (nav['PRICE_DATE']>=20200201) & (nav['PRICE_DATE']<=20200230)]

,index,F_INFO_WINDCODE,ANN_DATE,PRICE_DATE,F_NAV_UNIT,F_NAV_DIVACCUMULATED,F_NAV_ADJFACTOR,F_PRT_NETASSET,F_ASSET_MERGEDSHARESORNOT,NETASSET_TOTAL,F_NAV_ADJUSTED,IS_EXDIVIDENDDATE,F_NAV_DISTRIBUTION,S_INFO_ASHARECODE,CUM_NET_ASSET_VALUE,main_type,1M,return
13458368,4282825,920003.OF,2020-02-04,20200203,1.7713,NaN,1.519726,NaN,0,NaN,2.691890,0.0,0.48,S11647371,2.2513,Equity Fund,True,-0.077111
13458369,4282826,920003.OF,2020-02-05,20200204,1.8251,NaN,1.519726,NaN,0,NaN,2.773651,0.0,0.48,S11647371,2.3051,Equity Fund,True,0.030373
13458370,4282827,920003.OF,2020-02-06,20200205,1.8321,NaN,1.519726,NaN,0,NaN,2.784289,0.0,0.48,S11647371,2.3121,Equity Fund,True,0.003835
13458371,4282828,920003.OF,2020-02-07,20200206,1.8780,NaN,1.519726,NaN,0,NaN,2.854045,0.0,0.48,S11647371,2.3580,Equity Fund,True,0.025053
13458372,4282829,920003.OF,2020-02-08,20200207,1.9025,NaN,1.519726,NaN,0,NaN,2.891278,0.0,0.48,S11647371,2.3825,Equity Fund,True,0.013046
13458373,4282830,920003.OF,2020-02-11,20200210,1.9169,NaN,1.519726,NaN,0,NaN,2.913162,0.0,0.48,S11647371,2.3969,Equity Fund,True,0.007569
13458374,4282831,920003.OF,2020-02-12,20200211,1.9236,NaN,1.519726,NaN,0,NaN,2.923344,0.0,0.48,S11647371,2.4036,Equity Fund,True,0.003495
13458375,4282832,920003.OF,2020-02-13,20200212,1.9665,NaN,1.519726,NaN,0,NaN,2.988540,0.0,0.48,S11647371,2.4465,Equity Fund,True,0.022302
13458376,4282833,920003.OF,2020-02-14,20200213,1.9823,NaN,1.519726,NaN,0,NaN,3.012552,0.0,0.48,S11647371,2.4623,Equity Fund,True,0.008035
13458377,4282834,920003.OF,2020-02-15,20200214,1.9994,NaN,1.519726,NaN,0,NaN,3.038539,0.0,0.48,S11647371,2.4794,Equity Fund,True,0.008626
